In [1]:
import pandas as pd
import numpy as np

# Path to the CSV file
file_path = 'https://raw.githubusercontent.com/veghmark12/r_d_investments/refs/heads/main/df_with_closest_fua_investorclusters.csv'
file_path2 = 'https://raw.githubusercontent.com/veghmark12/r_d_investments/refs/heads/main/investor_coordinates.csv'
file_path3 = 'https://raw.githubusercontent.com/veghmark12/r_d_investments/refs/heads/main/top100_acc_inc_coordinates.csv'
file_path4 = 'https://raw.githubusercontent.com/veghmark12/r_d_investments/refs/heads/main/top100_university_coordinates.csv'
# Load the CSV into a DataFrame
df1 = pd.read_csv(file_path)
df2 = pd.read_csv(file_path2)
df3 = pd.read_csv(file_path3)
df4 = pd.read_csv(file_path4)

nan_value = float("NaN")

df1.rename(columns={'Closest FUA': 'FUA'}, inplace=True)

df = df3
df

,#,Investor Name,HQ Country,HQ City,Latitude,Longitude
0,1,+Impact,Sweden,Stockholm,59.325117,18.071093
1,2,1Kubator,France,Lyon,45.757814,4.832011
2,3,5starts,Austria,Vienna,48.208354,16.372504
3,4,21st by CentraleSupélec,France,Gif-Sur-Yvette,48.701882,2.134529
4,5,70Ventures,Lithuania,Vilnius,54.687046,25.282911
...,...,...,...,...,...,...
370,410,Xpreneurs,Germany,Munich,48.137108,11.575382
371,412,Zagatub,France,Paris,48.853495,2.348391
372,413,ZERO,Italy,Rome,41.893320,12.482932
373,414,Zero Acceleratore,Italy,Rome,41.893320,12.482932


In [2]:
!pip install opencage
!pip install geopy pandas

In [3]:
df

,#,Investor Name,HQ Country,HQ City,Latitude,Longitude
0,1,+Impact,Sweden,Stockholm,59.325117,18.071093
1,2,1Kubator,France,Lyon,45.757814,4.832011
2,3,5starts,Austria,Vienna,48.208354,16.372504
3,4,21st by CentraleSupélec,France,Gif-Sur-Yvette,48.701882,2.134529
4,5,70Ventures,Lithuania,Vilnius,54.687046,25.282911
...,...,...,...,...,...,...
370,410,Xpreneurs,Germany,Munich,48.137108,11.575382
371,412,Zagatub,France,Paris,48.853495,2.348391
372,413,ZERO,Italy,Rome,41.893320,12.482932
373,414,Zero Acceleratore,Italy,Rome,41.893320,12.482932


In [4]:
import pandas as pd

In [5]:
# Remove leading and trailing spaces from the 'country' column
df['Country'] = df['Country'].str.strip()

KeyError: 'Country'

In [ ]:
# List of the 27 EU countries
eu_countries = ['Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czech Republic',
 'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece',
 'Hungary', 'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg',
 'Malta', 'Netherlands', 'Poland', 'Portugal', 'Romania', 'Slovakia', 'Slovak Republic',
 'Slovenia', 'Spain', 'Sweden']

In [ ]:
# Filter to keep only EU countries
df = df[df['Country'].isin(eu_countries)]

In [ ]:
df

In [ ]:
import pandas as pd
from math import radians, sin, cos, sqrt, atan2

# Load your dataframe
fuas_file = 'EU_FUAs_coordinates.csv'  # Replace with the FUAs file

fuas_df = pd.read_csv(fuas_file)  # FUAs with coordinates

# Haversine formula to calculate the great-circle distance between two points
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in kilometers
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

# Find the closest FUA for each city
def find_closest_fua(city_lat, city_lon, fua_data):
    min_distance = float('inf')
    closest_fua = None
    for _, fua_row in fua_data.iterrows():
        fua_lat = fua_row['Latitude']
        fua_lon = fua_row['Longitude']
        distance = haversine(city_lat, city_lon, fua_lat, fua_lon)
        if distance < min_distance:
            min_distance = distance
            closest_fua = fua_row['FUA']
    return closest_fua

# Assuming your dataframe has 'City', 'Latitude', and 'Longitude' columns
df['Closest FUA'] = df.apply(
    lambda row: find_closest_fua(row['Latitude'], row['Longitude'], fuas_df), axis=1
)

# Save the updated dataframe to a new file
output_file = 'universities_closest_FUA.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
df

In [ ]:
import pandas as pd
import folium

# Load the data
fuas_file = 'FUAs_with_coordinates.csv'  # Replace with your FUAs file

fuas_df = pd.read_csv(fuas_file)  # FUAs with coordinates

# Count occurrences of each FUA in the "Closest FUA" column
fua_counts = df['Closest FUA'].value_counts().reset_index()
fua_counts.columns = ['FUA', 'Count']

# Merge with FUAs to get coordinates
fua_counts = fua_counts.merge(fuas_df, on='FUA', how='left')

# Initialize a map
m = folium.Map(location=[50, 10], zoom_start=5)  # Centered in Europe

# Calculate the total sum of counts
total_count = fua_counts['Count'].sum()

# Add circles to the map
for _, row in fua_counts.iterrows():
    percentage = row['Count'] / total_count  # Calculate the percentage of the total count
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=percentage ** 0.5 * 50,  # Circle size proportional to percentage (adjust multiplier as needed)
        popup=f"{row['FUA']}: {row['Count']} ({percentage:.2%})",
        color='purple',
        fill=True,
        fill_color='purple'
    ).add_to(m)

# Save map to an HTML file
output_map = 'fuas_map.html'
m.save(output_map)
print(f"Map saved to {output_map}")

output_file = 'scaleups_count.csv'
fua_counts.to_csv(output_file, index=False)

fua_counts